# 02 · Building a coded wavefront sensor from optical components

A **coded wavefront sensor** (CWFS) puts a thin random phase mask a short distance *d* in front of
a camera [1, 2]. A plane wave makes a speckle pattern on the camera: the **reference**. A
distorted wavefront W(x, y) makes the same speckle, locally shifted by

$$\mathbf{s}(x, y) = d\,\nabla W(x, y),$$

where W is the optical path difference (OPD) of the wave arriving at the mask. Measuring the shifts
gives ∇W, and integrating gives W.

This notebook builds a CWFS behind a microscope from OptiVolume's components, simulates two beads,
and looks at every stage:
1. the scene and the field leaving it;
2. the sensor assembled from components, with the sampling checked at every step;
3. the camera images: reference and object speckle;
4. the field arriving at the phase mask: amplitude and OPD;
5. the OPD gradients, and the speckle shift d·∇W they predict;
6. a check that the object speckle really is the reference shifted by d·∇W.

The settings follow the previous coded-wavefront-sensing example: λ = 500 nm, 100 nm voxels, NA 0.7, 40×
magnification, 12 µm mask tiles 100 nm high, mask 1.43 mm in front of a camera with 6 µm pixels.
It takes one to two minutes on a CPU; `DEVICE` picks a GPU when there is one.

In [ ]:
import math

import numpy as np
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt

from optical_volume import optics, scene

torch.set_grad_enabled(False)          # simulation only; see the fitting examples for gradients
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Light and object space
WL = 500e-9                        # vacuum wavelength (m)
N_MEDIUM = 1.51                    # immersion medium
OBJ_SPACING = (100e-9,) * 3        # voxel size (m)
SHAPE = (600, 600, 100)            # 60 x 60 x 10 µm

# Microscope
NA = 0.7
MAG = 40                           # physical magnification
RESAMPLE = 2                       # numerical: 2x more samples in image space
IMG_SPACING = (OBJ_SPACING[0] * MAG / RESAMPLE,) * 2   # 2 µm on the camera side

# Coded wavefront sensor
MASK_DISTANCE = 1.43e-3            # d: mask in front of the camera (m)
TILE, HEIGHT, N_MASK, EDGE = 12e-6, 100e-9, 1.46, 0.6   # tile side, height, index, edge blur (samples)
PIXEL = 6e-6                       # camera pixel (m)
PAD = 512                          # propagation padding (samples)


def show(ax, img, spacing, title, units=1e-6, **kw):
    """Plot an array indexed [x, y] with x to the right and y up; axes in µm."""
    img = np.asarray(img)
    ext = [0, img.shape[0] * spacing / units, 0, img.shape[1] * spacing / units]
    im = ax.imshow(img.T, origin="lower", extent=ext, **kw)
    ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)"); ax.set_title(title, fontsize=9)
    return im

## 1. The scene: two beads

Two beads (n = 1.46, radii 5 µm and 3 µm) in an immersion medium (n = 1.51). The beam propagation
method (BPM) carries a plane wave through the volume, slice by slice along +z.

In [ ]:
vol = scene.CollectionVolume(SHAPE, OBJ_SPACING, N_MEDIUM)
vol.add([scene.Sphere(torch.tensor((20e-6, 45e-6, 3e-6)), torch.tensor(5e-6), torch.tensor(1.46)),
         scene.Sphere(torch.tensor((30e-6, 20e-6, 5e-6)), torch.tensor(3e-6), torch.tensor(1.46))])
bpm = optics.BeamPropagation(WL, OBJ_SPACING, N_MEDIUM)
vol.to(DEVICE); bpm.to(DEVICE)
n = vol()
exit_field = bpm(n).to(torch.complex64)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = show(ax[0], n[:, :, 40].cpu(), OBJ_SPACING[0], "refractive index, slice z = 4 µm", cmap="viridis")
fig.colorbar(im, ax=ax[0])
im = show(ax[1], np.angle(exit_field.cpu()), OBJ_SPACING[0], "phase of the field leaving the volume (rad)", cmap="twilight")
fig.colorbar(im, ax=ax[1])
plt.tight_layout()

## 2. Assembling the sensor from components

The camera sits in the image plane of the microscope, and the mask a distance d in front of it. To
simulate that, the image-plane field is propagated **back** by d to the mask, multiplied by the
mask's transmission, and propagated **forward** by d to the camera. Both propagators share one
distance parameter, so fitting it moves the mask as one physical distance.

The chain, grouped into three parts so that the intermediate results can be recorded:

| part | element | what it does |
|---|---|---|
| to the mask | `AngularSpectrum` (backward, in the medium) | refocus from the exit face to the middle of the volume |
| | `FourFSystem` | microscope: pupil of NA 0.7 |
| | `Magnification(40)` | physical: spacing × 40, amplitude ÷ 40 |
| | `Resample(2)` | numerical: twice the samples (2 µm) |
| | `AngularSpectrum(−d)` | from the image plane back to the mask |
| through the mask | `BinaryPhaseMask` | random binary tiles, phase step 2π(n_mask − 1)h/λ |
| | `AngularSpectrum(+d)` | from the mask to the camera |
| | `Intensity` | \|E\|² |
| camera | `PixelIntegrator` | sums the intensity over each 6 µm pixel |

In [ ]:
refocus = optics.AngularSpectrum(WL / N_MEDIUM, OBJ_SPACING[:2], SHAPE[2] * OBJ_SPACING[2] / 2,
                                 direction="backward", padding=PAD)
lens = optics.FourFSystem(WL, OBJ_SPACING[:2], SHAPE[:2], NA)
to_mask = optics.AngularSpectrum(WL, IMG_SPACING, MASK_DISTANCE, direction="backward", padding=PAD)
mask = optics.BinaryPhaseMask(WL, TILE, IMG_SPACING, [s * RESAMPLE for s in SHAPE[:2]], HEIGHT, EDGE, N_MASK, seed=0)
to_camera = optics.AngularSpectrum(WL, IMG_SPACING, MASK_DISTANCE, direction="forward", padding=PAD)
to_camera.distance = to_mask.distance          # one physical distance, shared

front = optics.OpticalSystem((refocus, lens, optics.Magnification(MAG), optics.Resample(RESAMPLE), to_mask))
back = optics.OpticalSystem((mask, to_camera, optics.Intensity()))
camera = optics.PixelIntegrator(pixel_size=PIXEL, spacing=IMG_SPACING)
cwfs = optics.OpticalSystem((front, back, camera), record=True).to(DEVICE)

print(f"phase step of the mask: {2 * math.pi * (N_MASK - 1) * HEIGHT / WL:.2f} rad")
print(f"{'stage':28} {'sample spacing (µm)':>20}")
s = OBJ_SPACING[:2]
for name, element in [("object (exit face)", None), ("refocus", refocus), ("4f lens", lens),
                      ("Magnification(40)", front.elements[2]), ("Resample(2)", front.elements[3]),
                      ("back to the mask", to_mask), ("mask, forward, |E|²", back), ("camera pixels", camera)]:
    if element is not None and hasattr(element, "output_spacing"):
        s = element.output_spacing(s)
    print(f"{name:28} {s[0] * 1e6:20.2f}")
print("whole sensor:", [v * 1e6 for v in cwfs.output_spacing(OBJ_SPACING[:2])], "µm")

## 3. Camera images: reference and object speckle

The **reference** is the image of an empty field (a plane wave). The **object** image has the same
speckle, moved where the beads tilt the wavefront and modulated in brightness. Both are shown
normalised by the reference maximum. Overview images undersample the speckle, so the bottom row
shows a 1:1 crop (one screen pixel per camera pixel) around the larger bead.

In [ ]:
ref = cwfs(torch.ones(SHAPE[:2], dtype=torch.complex64, device=DEVICE))
mask_ref, sensor_ref, _ = [f.cpu() for f in cwfs.fields]          # mask-plane field, |E|² at 2 µm, camera
obj = cwfs(exit_field)
mask_obj, sensor_obj, _ = [f.cpu() for f in cwfs.fields]
ref, obj = ref.cpu(), obj.cpu()
scale = ref.max()

fig, ax = plt.subplots(2, 3, figsize=(15, 9.5))
kw = dict(cmap="gray", vmin=0, vmax=1.2)
for a, img, ttl in zip(ax[0], (ref / scale, obj / scale), ("reference speckle (camera)", "object speckle (camera)")):
    fig.colorbar(show(a, img, PIXEL, ttl, **kw), ax=a, shrink=0.8)
fig.colorbar(show(ax[0, 2], (obj - ref) / scale, PIXEL, "object − reference", cmap="RdBu_r", vmin=-0.6, vmax=0.6),
             ax=ax[0, 2], shrink=0.8)
crop = (slice(100, 180), slice(255, 335))                         # 480 µm around the larger bead
for a, img, ttl in zip(ax[1], (ref, obj, obj - ref), ("reference, 1:1 crop", "object, 1:1 crop", "difference, 1:1 crop")):
    a.imshow((img[crop] / scale).T, origin="lower", interpolation="nearest",
             cmap="RdBu_r" if "diff" in ttl else "gray", vmin=-0.6 if "diff" in ttl else 0, vmax=0.6 if "diff" in ttl else 1.2)
    a.set_title(ttl, fontsize=9); a.set_xlabel("x (pixels)"); a.set_ylabel("y (pixels)")
plt.tight_layout()
print(f"camera image: {tuple(ref.shape)} pixels of {PIXEL * 1e6:g} µm; speckle contrast (std/mean) of the reference: "
      f"{(ref.std() / ref.mean()).item():.2f}")

## 4. The field arriving at the phase mask: amplitude and OPD

The recorded field after the first part is the wave arriving at the mask, in the camera's (image)
space. Dividing it by the reference field removes everything the two have in common, leaving the
object's effect, Q = U_obj / U_ref, with a constant background phase removed:
- **amplitude |Q|:** bright and dark rings, because the mask plane is 1.43 mm from focus;
- **OPD W = arg(Q)·λ/2π:** the wavefront the mask sees.

arg(Q) wraps at ±π, i.e. at OPD ±λ/2, and the larger bead delays by more than that. So W is computed
from its gradients (section 5), which are taken from the phase difference between neighbouring
samples and never wrap, and integrated with a least-squares Fourier method [3]. Wrapping W again
reproduces arg(Q).

The OPD is negative: the beads have a lower index (1.46) than the medium (1.51).

In [ ]:
k = 2 * math.pi / WL
dx = IMG_SPACING[0]
Q = (mask_obj / mask_ref).numpy()
background = np.angle(np.mean(Q[:50, :50] / np.abs(Q[:50, :50])))     # a corner far from the beads
Q = Q * np.exp(-1j * background)


def opd_gradients(Q, dx):
    """dW/dx, dW/dy from phase differences of neighbouring samples (no wrapping), central differences."""
    gx = np.zeros(Q.shape); gy = np.zeros(Q.shape)
    gx[1:-1] = np.angle(Q[2:] * np.conj(Q[:-2])) / (2 * dx) / k
    gy[:, 1:-1] = np.angle(Q[:, 2:] * np.conj(Q[:, :-2])) / (2 * dx) / k
    return gx, gy


def integrate(gx, gy, dx):
    """Least-squares surface with these gradients (Frankot-Chellappa, periodic), zero mean."""
    f = np.fft.fftfreq(gx.shape[0], dx)
    FX, FY = np.meshgrid(f, np.fft.fftfreq(gx.shape[1], dx), indexing="ij")
    den = FX**2 + FY**2; den[0, 0] = 1
    W = np.real(np.fft.ifft2(-1j * (FX * np.fft.fft2(gx) + FY * np.fft.fft2(gy)) / (2 * math.pi * den)))
    return W - np.median(W)


gx, gy = opd_gradients(Q, dx)
W = integrate(gx, gy, dx)
rewrap_error = np.angle(np.exp(1j * k * W) * np.conj(np.exp(1j * np.angle(Q))))[50:-50, 50:-50]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
fig.colorbar(show(ax[0], np.abs(Q), dx, "amplitude at the mask, |U_obj| / |U_ref|", cmap="gray"), ax=ax[0], shrink=0.85)
fig.colorbar(show(ax[1], np.angle(Q), dx, "phase at the mask, arg Q (rad, wrapped)", cmap="twilight", vmin=-math.pi, vmax=math.pi),
             ax=ax[1], shrink=0.85)
fig.colorbar(show(ax[2], W * 1e9, dx, "OPD at the mask, W (nm)", cmap="viridis"), ax=ax[2], shrink=0.85)
plt.tight_layout()
print(f"OPD from {W.min() * 1e9:.0f} to {W.max() * 1e9:.0f} nm; wrapping W again matches arg Q to "
      f"{np.sqrt(np.mean(rewrap_error**2)):.4f} rad rms")

## 5. OPD gradients and the speckle shift they predict

The local tilt of the wavefront at the mask is ∇W. Over the distance d to the camera it moves the
speckle by **s = d·∇W**. The arrows show s (every 40th sample); slopes of 10 mrad move the speckle
by about 14 µm, more than two camera pixels.

The gradients are in the camera's space. In the object's space they are 40× larger (the
magnification): the sensor measures the object's OPD gradient divided by the magnification.

In [ ]:
sx, sy = MASK_DISTANCE * gx, MASK_DISTANCE * gy                 # speckle shift (m)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
lim = 1e3 * max(np.abs(gx).max(), np.abs(gy).max())
fig.colorbar(show(ax[0], gx * 1e3, dx, "∂W/∂x (mrad)", cmap="RdBu_r", vmin=-lim, vmax=lim), ax=ax[0], shrink=0.85)
fig.colorbar(show(ax[1], gy * 1e3, dx, "∂W/∂y (mrad)", cmap="RdBu_r", vmin=-lim, vmax=lim), ax=ax[1], shrink=0.85)
mag = np.hypot(sx, sy) * 1e6
fig.colorbar(show(ax[2], mag, dx, "speckle shift |d·∇W| (µm), arrows: d·∇W", cmap="magma"), ax=ax[2], shrink=0.85)
step = 40
ii, jj = np.meshgrid(np.arange(0, Q.shape[0], step), np.arange(0, Q.shape[1], step), indexing="ij")
ax[2].quiver(ii * dx * 1e6, jj * dx * 1e6, sx[::step, ::step], sy[::step, ::step], color="c",
             angles="xy", scale_units="xy", scale=1e-6 / 10)        # arrows drawn 10x longer
plt.tight_layout()
print(f"largest slope {lim:.1f} mrad -> largest shift {mag.max():.1f} µm = {mag.max() / (PIXEL * 1e6):.1f} camera pixels")

## 6. Is the object speckle the shifted reference?

If the CWFS principle holds, moving the reference pattern by s = d·∇W should reproduce the object
pattern. Below, on the full-resolution sensor intensity (2 µm samples) around the beads, the
correlation with the object pattern of:
- the reference as it is;
- the reference shifted by +d·∇W, and by −d·∇W (to check the direction);
- the shifted reference times |Q|², the brightness change seen at the mask.

The shift alone explains part of the change; with the brightness it explains most of it. The figure
compares what is left over: the object minus the reference, and the object minus the prediction,
on the same colour scale, next to the shift magnitude. The rest
comes from the beads' curvature: where the gradient changes over the region one mask tile spreads
its light across, λd/a ≈ 60 µm, neighbouring parts of the speckle move by different amounts and the
pattern partly changes instead of only moving.

In [ ]:
def shift_image(img, sx, sy, dx):
    """img(x - s(x)): the image moved by the shift field s (bilinear backward map)."""
    n0, n1 = img.shape
    ii, jj = np.meshgrid(np.arange(n0), np.arange(n1), indexing="ij")
    pi, pj = ii - sx / dx, jj - sy / dx
    grid = torch.tensor(np.stack([pj / (n1 - 1) * 2 - 1, pi / (n0 - 1) * 2 - 1], axis=-1)[None], dtype=torch.float32)
    return F.grid_sample(torch.as_tensor(img, dtype=torch.float32)[None, None], grid, align_corners=True)[0, 0].numpy()


I_ref, I_obj = sensor_ref.numpy(), sensor_obj.numpy()
region = np.hypot(sx, sy) > 0.3 * dx                     # where the beads move the speckle
corr = lambda a: np.corrcoef(a[region], I_obj[region])[0, 1]
cases = {"reference as it is": I_ref,
         "reference shifted by +d·∇W": shift_image(I_ref, sx, sy, dx),
         "reference shifted by −d·∇W": shift_image(I_ref, -sx, -sy, dx),
         "shifted by +d·∇W, times |Q|²": shift_image(I_ref, sx, sy, dx) * np.abs(Q)**2}
for name, img in cases.items():
    print(f"{name:32} correlation with the object pattern: {corr(img):.3f}")

c = (slice(280, 520), slice(780, 1020))                  # 480 µm around the larger bead
pred = cases["shifted by +d·∇W, times |Q|²"]
vmax = np.percentile(np.abs(I_obj - I_ref)[c], 99)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, img, ttl in zip(ax, (I_obj - I_ref, I_obj - pred, np.hypot(sx, sy) * 1e6),
                       ("object − reference", "object − prediction", "|d·∇W| (µm)")):
    im = a.imshow(img[c].T, origin="lower", interpolation="nearest",
                  cmap="magma" if "|d" in ttl else "RdBu_r", vmin=0 if "|d" in ttl else -vmax, vmax=None if "|d" in ttl else vmax)
    a.set_title(f"{ttl}, 2 µm samples", fontsize=9); a.set_xlabel("x (samples)"); a.set_ylabel("y (samples)")
    fig.colorbar(im, ax=a, shrink=0.85)
plt.tight_layout()

## What next

- `01_scene_basics`: how scenes, rotations and refractive-index maps are set up.
- Fitting parameters of this chain (bead size, mask distance, ...) from images, generating training
  data with it, and imaging voxel objects: planned examples 03–06.
- `pipelines.CodedWavefrontSensor` packages the part after the microscope; its settings become
  configurable in a later release.

## References

1. C. Wang, X. Dun, Q. Fu, W. Heidrich, "Ultra-high resolution coded wavefront sensor,"
   *Opt. Express* **25**, 13736–13746 (2017).
2. P. Berto, H. Rigneault, M. Guillon, "Wavefront sensing with a thin diffuser," *Opt. Lett.* **42**,
   5117–5120 (2017).
3. R. T. Frankot, R. Chellappa, "A method for enforcing integrability in shape from shading
   algorithms," *IEEE Trans. Pattern Anal. Mach. Intell.* **10**, 439–451 (1988).